In [1]:
!pip install langchain langchain-openai langchain-community pydantic pypdf python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

In [2]:
import os
from typing import Literal, List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

In [3]:
os.environ["OPENROUTER_API_KEY"] = "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"

In [4]:
MODEL_NAME = "nvidia/nemotron-3-nano-30b-a3b:free"

low_temp_model = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0.1,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

high_temp_model = ChatOpenAI(
    model=MODEL_NAME,
    temperature=1.2,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

In [5]:
prompt = """
Explain what machine learning is.
Make it short.
Be poetic in your response.
Format your answer as JSON with this structure:
{
  "answer": "...",
  "style": "..."
}
"""

low_response = low_temp_model.invoke(prompt)
high_response = high_temp_model.invoke(prompt)

print("Low temperature response:")
print(low_response.content)

print("\nHigh temperature response:")
print(high_response.content)

Low temperature response:
{
  "answer": "Machinelearning is a garden where data seeds sprout patterns, learning to predict the next bloom.",
  "style": "poetic"
}

High temperature response:
{
  "answer": "A shy algorithm that learns the song of data, shaping clouds of insight from silent noise.",
  "style": "poetic"
}


In [6]:
class SentimentResult(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        description="The sentiment of the input sentence."
    )

sentiment_model = low_temp_model.with_structured_output(SentimentResult)

sentences = [
    "Kindness creates lasting joy.",
    "Success rewards persistent effort.",
    "I love Sunlight. It warms the skin.",
    "Pessemestic all the time.",
    "The storm caused damage!",
    "The clock ticks steadily.",
]

for sentence in sentences:
    result = sentiment_model.invoke(sentence)
    print(sentence, "->", result.sentiment)

Kindness creates lasting joy. -> positive
Success rewards persistent effort. -> positive
I love Sunlight. It warms the skin. -> positive
Pessemestic all the time. -> positive
The storm caused damage! -> neutral
The clock ticks steadily. -> neutral


The structured output worked because the model only returned one of the allowed labels: positive, neutral, or negative.

However, the model did not classify all examples correctly. It labeled "Pessemestic all the time." as positive, even though it should likely be negative. It also labeled "The storm caused damage!" as neutral, although damage usually suggests a negative sentiment.

This shows that structured output controls the format of the answer, but it does not guarantee that the model's classification is correct.

In [7]:
class CategoryResult(BaseModel):
    tags: List[Literal["cars", "shopping", "sports", "study", "work"]] = Field(
        description="One or more relevant categories for the input text."
    )

category_model = low_temp_model.with_structured_output(CategoryResult)

texts = [
    "That restoration looks incredible; you have a real talent for mechanics.",
    "I found the perfect gift today! The staff was incredibly helpful.",
    "Great game today! Your teamwork was the key to that victory.",
    "Learning together helped me finally grasp these concepts. Thank you!",
]

for text in texts:
    prompt = f"""
Classify the sentence into one or more of these tags only:
cars, shopping, sports, study, work.

Rules:
- cars: vehicles, mechanics, engines, restoration, driving
- shopping: gifts, stores, staff, buying, products
- sports: games, teams, victory, matches, players
- study: learning, concepts, school, studying, understanding
- work: job, office, business, projects, professional tasks

Sentence:
{text}
"""
    result = category_model.invoke(prompt)
    print(text, "->", result.tags)

That restoration looks incredible; you have a real talent for mechanics. -> ['cars']
I found the perfect gift today! The staff was incredibly helpful. -> ['shopping']
Great game today! Your teamwork was the key to that victory. -> ['sports']
Learning together helped me finally grasp these concepts. Thank you! -> ['study']


The categorization model successfully returned tags from the allowed list only: cars, shopping, sports, study, and work.

Each sentence was classified correctly based on its main meaning. The restoration and mechanics sentence was tagged as cars, the gift sentence as shopping, the game sentence as sports, and the learning sentence as study.

This shows that structured output can be useful for controlled multi-label classification, because the model returns a list of valid tags instead of free text.


In [8]:
from langchain_community.document_loaders import PyPDFLoader

In [9]:
loader = PyPDFLoader("Abdulaziz_Almousa_Resume.pdf")
documents = loader.load()

cv_text = "\n\n".join(doc.page_content for doc in documents)

print(cv_text[:1000])

Abdulaziz Almousa
almousaabdallaziz@gmail.com
 
+966 546564821
 
Abdulaziz Almousa
 
Abdulaziz Almousa
 
PROFILE
Software Engineering graduate with IT Audit experience at Deloitte and Business Analysis experience at Tahakom, working with enterprise-scale 
data and national digital platforms. Transforms large, fragmented datasets into reliable inputs and translates ambiguous requirements into 
structured workflows that support business and system decisions. Focused on strategy and transformation roles improving how data, 
processes, and decisions connect in practice.
PROFESSIONAL EXPERIENCE
IT Audit Intern
Deloitte Middle East
•Processed and reconciled General Ledger and Journal Entry datasets of up to 9M+ records across 15+ engagements, 
ensuring audit testing was performed on complete and reliable data
10/2025 – 03/2026
Riyadh, Saudi Arabia
•Identified and resolved inconsistencies across fragmented data sources before testing began, reducing downstream 
review iterations and keeping e

In [10]:
class ExperienceItem(BaseModel):
    company: str = Field(description="Company name")
    role: str = Field(description="Job title or role")
    duration: str = Field(description="Time period or duration")
    responsibilities: List[str] = Field(description="Main responsibilities or achievements")


class ResumeExtraction(BaseModel):
    candidate_name: str = Field(description="Candidate full name")
    email: str | None = Field(description="Email address if found")
    phone: str | None = Field(description="Phone number if found")
    education: List[str] = Field(description="Education history")
    skills: List[str] = Field(description="Technical and professional skills")
    experience: List[ExperienceItem] = Field(description="Work experience")
    projects: List[str] = Field(description="Projects mentioned in the resume")

In [11]:
resume_model = low_temp_model.with_structured_output(ResumeExtraction)

In [12]:
resume_prompt = f"""
Extract the key information from this resume.

Return the answer using the required structured schema.

Resume text:
{cv_text}
"""

resume_result = resume_model.invoke(resume_prompt)

print(resume_result)

candidate_name='Abdulaziz Almousa' email='almousaabdallaziz@gmail.com' phone='+966 546564821' education=['Software Engineering with Honors, King Saud University, GPA: 4.62, 09/2021 – 08/2025'] skills=['Technical: Excel, Data Analysis, Python, Requirement Management, Process Modeling (BPMN), SDLC, Agile, PowerPoint, Visio, Figma, Trello', 'Soft: Business Acumen, Strategic Thinking, Problem Solving, Critical Thinking, Teamwork, Leadership, Stakeholder Engagement, Communication, Adaptability, Learning Agility, Results Orientation, Time Management, Analytical Thinking'] experience=[ExperienceItem(company='Deloitte Middle East', role='IT Audit Intern', duration='10/2025 – 03/2026', responsibilities=['Processed and reconciled General Ledger and Journal Entry datasets of up to 9M+ records across 15+ engagements, ensuring audit testing was performed on complete and reliable data', 'Identified and resolved inconsistencies across fragmented data sources before testing began, reducing downstream 

In [13]:
print("Candidate Name:", resume_result.candidate_name)
print("Email:", resume_result.email)
print("Phone:", resume_result.phone)

print("\nEducation:")
for edu in resume_result.education:
    print("-", edu)

print("\nSkills:")
for skill in resume_result.skills:
    print("-", skill)

print("\nExperience:")
for exp in resume_result.experience:
    print("Company:", exp.company)
    print("Role:", exp.role)
    print("Duration:", exp.duration)
    print("Responsibilities:")
    for r in exp.responsibilities:
        print("-", r)
    print()

print("\nProjects:")
for project in resume_result.projects:
    print("-", project)

Candidate Name: Abdulaziz Almousa
Email: almousaabdallaziz@gmail.com
Phone: +966 546564821

Education:
- Software Engineering with Honors, King Saud University, GPA: 4.62, 09/2021 – 08/2025

Skills:
- Technical: Excel, Data Analysis, Python, Requirement Management, Process Modeling (BPMN), SDLC, Agile, PowerPoint, Visio, Figma, Trello
- Soft: Business Acumen, Strategic Thinking, Problem Solving, Critical Thinking, Teamwork, Leadership, Stakeholder Engagement, Communication, Adaptability, Learning Agility, Results Orientation, Time Management, Analytical Thinking

Experience:
Company: Deloitte Middle East
Role: IT Audit Intern
Duration: 10/2025 – 03/2026
Responsibilities:
- Processed and reconciled General Ledger and Journal Entry datasets of up to 9M+ records across 15+ engagements, ensuring audit testing was performed on complete and reliable data
- Identified and resolved inconsistencies across fragmented data sources before testing began, reducing downstream review iterations and ke

The model extracted structured information from the resume, including the candidate name, contact details, education, skills, work experience, and projects.

Using `.with_structured_output()` helped force the model to return the information in a consistent schema instead of a normal paragraph.

Some fields may still need manual checking because the quality of the extraction depends on how clearly the PDF text was loaded.